# Portfolio edition
This is a source-only publishing copy of the original coursework notebook. Saved outputs and account-specific notebook metadata were cleared. Run the notebook with the documented data and compute requirements; historical results are not represented as a fresh execution.


# 42028: Deep Learning and Convolutional Neural Networks
## Assignment 2 – Part A: Image Classification
---
**Name:** Vishnu Bhaiya  
**Dataset:** Dogs (10 Classes)  


---

### Experiments

**Experiment 1 – Baseline ResNet50 (pre-trained, fine-tuned)**
Standard ResNet50 with ImageNet pre-trained weights is fine-tuned on the dog breed
dataset. Transfer learning is used to initialise weights; all layers are then
re-trained on the dataset.

**Experiment 2 – Customized ResNet50 (+2 additional CONV layers)**
The same ResNet50 backbone is extended by adding **2 extra Conv2D blocks**
(Conv2D → BatchNorm → ReLU) after the backbone output and before the classification
head. This CONV-layer addition is directly motivated by the overfitting
observed in Experiment 1 and is the required CONV-layer customization.


---

**Folder structure — everything goes into `./image_classification/`:**
```
image_classification/
├── split_data/          ← train / val / test folders
│   ├── train/
│   ├── val/
│   └── test/
├── weights/
│   ├── baseline/        ← epoch checkpoints + best model
│   └── custom/          ← epoch checkpoints + best model
└── plots/               ← all saved figures
```

---

### Sections
1. Environment Setup
2. Imports
3. Configuration
4. Dataset Loading and Splitting
5. Data Visualisation
6. Image Data Generators and Shared Helpers
7. Experiment 1 – Baseline ResNet50
8. Experiment 2 – Customized ResNet50 (+2 CONV blocks)
9. Results Comparison and Discussion

---
## 1. Environment Setup

In [ ]:
!nvidia-smi

### Fix XLA / libdevice GPU error (SageMaker)
TensorFlow's XLA compiler needs `libdevice.10.bc`. Setting `XLA_FLAGS` before
importing TensorFlow prevents a GPU crash during `model.fit()`.

In [ ]:
!find / -type f -name "libdevice.10.bc" 2>/dev/null

In [ ]:
import os
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/opt/conda'
print("XLA_FLAGS set.")

In [ ]:
!pip install pydot graphviz -q
print("pydot and graphviz ready.")

---
## 2. Imports
All required packages imported in a single cell.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "opencv-python-headless", "-q",
                       "--quiet", "--no-warn-conflicts"])

import os
import glob
import shutil
import warnings
import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.models       import Model
from tensorflow.keras.layers       import (Conv2D, MaxPooling2D, Dense,
                                            BatchNormalization, Activation,
                                            Add, Input, GlobalAveragePooling2D,
                                            Dropout)
from tensorflow.keras.callbacks    import (ModelCheckpoint, EarlyStopping,
                                            ReduceLROnPlateau)
from tensorflow.keras.regularizers import l2
from tensorflow.keras.initializers import glorot_uniform
from tensorflow.keras.optimizers   import Adam
from tensorflow.keras.utils        import plot_model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics         import (classification_report,
                                      confusion_matrix,
                                      ConfusionMatrixDisplay)
from IPython.display         import Image as IPImage, display

warnings.filterwarnings('ignore')

print(f"TensorFlow  : {tf.__version__}")
print(f"Keras       : {keras.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU')}")


---
## 3. Configuration
All constants in one place.

In [ ]:
IC_BASE      = './image_classification'
SPLIT_PATH   = os.path.join(IC_BASE, 'split_data')
WEIGHTS_DIR  = os.path.join(IC_BASE, 'weights')
PLOTS_DIR    = os.path.join(IC_BASE, 'plots')
TRAIN_DIR    = os.path.join(SPLIT_PATH, 'train')
VAL_DIR      = os.path.join(SPLIT_PATH, 'val')
TEST_DIR     = os.path.join(SPLIT_PATH, 'test')
B_CKPT_DIR   = os.path.join(WEIGHTS_DIR, 'baseline')
C_CKPT_DIR   = os.path.join(WEIGHTS_DIR, 'custom')

DATASET_PATH = os.environ.get('CV_DOGS_DATA', './data/dog-breeds')

STUDENT_ID   = 14679308
IMG_H        = 224          
IMG_W        = 224
BATCH_SIZE   = 32
NUM_CLASSES  = 10
MAX_EPOCHS   = 150
LR           = 1e-4      
CUSTOM_LR    = 5e-5        
PATIENCE     = 30

CLASS_NAMES  = [
    'English_springer', 'Bouvier_des_Flandres', 'komondor',
    'groenendael', 'dingo', 'Italian_greyhound',
    'borzoi', 'cocker_spaniel', 'Saluki', 'basenji'
]
VALID_EXT    = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')

for d in [SPLIT_PATH, PLOTS_DIR, B_CKPT_DIR, C_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print("=" * 60)
print("  CONFIGURATION SUMMARY")
print("=" * 60)
print(f"  Student ID (random_state) : {STUDENT_ID}")
print(f"  Image size                : {IMG_H} x {IMG_W}  (ResNet50 standard)")
print(f"  Batch size                : {BATCH_SIZE}")
print(f"  Number of classes         : {NUM_CLASSES}")
print(f"  Max epochs                : {MAX_EPOCHS}")
print(f"  Learning rate (baseline)  : {LR}")
print(f"  Learning rate (custom)    : {CUSTOM_LR}")
print(f"  Early stopping patience   : {PATIENCE}")
print(f"  Optimiser                 : Adam")
print(f"  Loss function             : Categorical Cross-Entropy")
print(f"  Transfer learning         : ResNet50 (ImageNet weights, all layers trained)")
print(f"  Data augmentation         : rotation 20, shift 0.2, flip, zoom 0.2")
print(f"  Dataset path              : {DATASET_PATH}")
print(f"  Output root               : {IC_BASE}/")
print("=" * 60)

---
## 4. Dataset Loading and Splitting
Dataset is **not pre-split**. Per the specification it must be split using
**Split:** 70% train / 15% val / 15% test (stratified).

In [ ]:
all_images, all_labels = [], []

print(f"{'Class':<30} {'Count':>6}")
print("-" * 38)
for cls in sorted(os.listdir(DATASET_PATH)):
    cls_dir = os.path.join(DATASET_PATH, cls)
    if not os.path.isdir(cls_dir):
        continue
    imgs = [os.path.join(cls_dir, f)
            for f in os.listdir(cls_dir) if f.endswith(VALID_EXT)]
    all_images.extend(imgs)
    all_labels.extend([cls] * len(imgs))
    print(f"  {cls:<28} {len(imgs):>6}")
print("-" * 38)
print(f"  {'TOTAL':<28} {len(all_images):>6}")

In [ ]:
X_train, X_tmp, y_train, y_tmp = train_test_split(
    all_images, all_labels, test_size=0.30,
    random_state=STUDENT_ID, stratify=all_labels)

X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50,
    random_state=STUDENT_ID, stratify=y_tmp)

total = len(all_images)
print(f"Dataset split  (random_state={STUDENT_ID})")
print("-" * 45)
print(f"  Training   : {len(X_train):>5}  ({len(X_train)/total*100:.1f}%)")
print(f"  Validation : {len(X_val):>5}  ({len(X_val)/total*100:.1f}%)")
print(f"  Test       : {len(X_test):>5}  ({len(X_test)/total*100:.1f}%)")
print(f"  Total      : {total:>5}  (100.0%)")

In [ ]:
def build_split_dirs(root, paths, labels, split):
    copied = 0
    for p, lbl in zip(paths, labels):
        dst_dir  = os.path.join(root, split, lbl)
        os.makedirs(dst_dir, exist_ok=True)
        dst_file = os.path.join(dst_dir, os.path.basename(p))
        if not os.path.exists(dst_file):
            shutil.copy2(p, dst_file)
            copied += 1
    return copied

if not os.path.exists(TRAIN_DIR) or len(os.listdir(TRAIN_DIR)) == 0:
    print("Copying images to split_data/ ...")
    n_tr = build_split_dirs(SPLIT_PATH, X_train, y_train, 'train')
    n_vl = build_split_dirs(SPLIT_PATH, X_val,   y_val,   'val')
    n_te = build_split_dirs(SPLIT_PATH, X_test,  y_test,  'test')
    print(f"  Copied: {n_tr} train | {n_vl} val | {n_te} test")
else:
    print("Split directories already exist – skipping copy.")
print(f"  train -> {TRAIN_DIR}")
print(f"  val   -> {VAL_DIR}")
print(f"  test  -> {TEST_DIR}")

---
## 5. Data Visualisation
### Sample images per class

In [ ]:
class_dirs = sorted(os.listdir(TRAIN_DIR))
fig, axes  = plt.subplots(2, 5, figsize=(16, 7))
axes       = axes.flatten()

for i, cls in enumerate(class_dirs):
    cls_path = os.path.join(TRAIN_DIR, cls)
    files    = [f for f in os.listdir(cls_path) if f.endswith(VALID_EXT)]
    if not files:
        axes[i].axis('off')
        continue
    img = cv2.imread(os.path.join(cls_path, files[0]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_W, IMG_H))
    axes[i].imshow(img)
    axes[i].set_title(cls.replace('_', ' '), fontsize=9, pad=4)
    axes[i].axis('off')
for j in range(len(class_dirs), len(axes)):
    axes[j].axis('off')

p = os.path.join(PLOTS_DIR, '00_sample_images.png')
plt.suptitle('Sample Image – One per Dog Breed Class  (Training Set)', fontsize=13)
plt.tight_layout()
plt.savefig(p, dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved: {p}")

In [ ]:
split_counts = {}
for split, split_dir in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    split_counts[split] = {}
    for cls in sorted(os.listdir(split_dir)):
        cls_dir = os.path.join(split_dir, cls)
        if os.path.isdir(cls_dir):
            split_counts[split][cls] = len(os.listdir(cls_dir))

x     = np.arange(NUM_CLASSES)
width = 0.25
labels_short = [c.replace('_', ' ')[:12] for c in sorted(split_counts['Train'].keys())]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width, list(split_counts['Train'].values()), width, label='Train')
ax.bar(x,         list(split_counts['Val'].values()),   width, label='Val')
ax.bar(x + width, list(split_counts['Test'].values()),  width, label='Test')
ax.set_xticks(x)
ax.set_xticklabels(labels_short, rotation=35, ha='right', fontsize=9)
ax.set_title('Class Distribution Across Train / Val / Test Splits', fontsize=12)
ax.set_ylabel('Number of Images')
ax.legend()
ax.grid(axis='y', alpha=0.3)
p = os.path.join(PLOTS_DIR, '01_class_distribution.png')
plt.tight_layout()
plt.savefig(p, dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved: {p}")

---
## 6. Image Data Generators and Shared Helpers

In [ ]:

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.2,
    fill_mode='nearest'
)
val_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

def make_generators():
    tg = train_datagen.flow_from_directory(
        TRAIN_DIR, target_size=(IMG_H, IMG_W), batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=True, seed=STUDENT_ID)
    vg = val_datagen.flow_from_directory(
        VAL_DIR, target_size=(IMG_H, IMG_W), batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False)
    eg = test_datagen.flow_from_directory(
        TEST_DIR, target_size=(IMG_H, IMG_W), batch_size=BATCH_SIZE,
        class_mode='categorical', shuffle=False)
    return tg, vg, eg

train_gen, val_gen, test_gen = make_generators()
STEPS  = len(X_train) // BATCH_SIZE
VSTEPS = len(X_val)   // BATCH_SIZE

print("Class index mapping:")
for cls, idx in sorted(train_gen.class_indices.items(), key=lambda x: x[1]):
    print(f"  {idx:>2}: {cls}")
print(f"\nSteps per epoch : {STEPS}  |  Val steps : {VSTEPS}")


### Transfer Learning – ResNet50

Both experiments use **ResNet50** as the backbone.

- Pre-trained weights from ImageNet are used to initialise the network.
- **All layers are re-trained** on the dog breed dataset (not frozen).

**Experiment 1 (Baseline):** Standard ResNet50 backbone + custom classification head.

**Experiment 2 (Custom):** Same ResNet50 backbone + **2 additional Conv2D blocks**
appended before the head. These extra CONV layers are the required customization.

In [ ]:
print("Using ResNet50 from tf.keras.applications (ImageNet weights).")

In [ ]:
print("Ready to build models.")

In [ ]:
def plot_history(history, title, save_path):
    acc, val_acc = history.history['accuracy'], history.history['val_accuracy']
    loss, val_loss = history.history['loss'], history.history['val_loss']
    ep = range(1, len(acc)+1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(ep, acc,     'bo-', lw=1.5, ms=3, label='Train Accuracy')
    ax1.plot(ep, val_acc, 'r-',  lw=2,          label='Val Accuracy')
    ax1.set_title(f'{title} – Accuracy');  ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy');  ax1.legend();  ax1.grid(True, alpha=0.3)

    ax2.plot(ep, loss,     'bo-', lw=1.5, ms=3, label='Train Loss')
    ax2.plot(ep, val_loss, 'r-',  lw=2,          label='Val Loss')
    ax2.set_title(f'{title} – Loss');  ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss');  ax2.legend();  ax2.grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

print("plot_history() ready.")

In [ ]:
def full_evaluation(model, generator, exp_name, plots_dir):
    generator.reset()
    y_prob  = model.predict(generator, verbose=1)
    y_pred  = np.argmax(y_prob, axis=1)
    y_true  = generator.classes
    labels  = list(generator.class_indices.keys())

    print(f"\nClassification Report – {exp_name}")
    print(classification_report(y_true, y_pred, target_names=labels, digits=4))

    # Confusion matrix
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=[l.replace('_',' ')[:10]
                                                       for l in labels])
    fig, ax = plt.subplots(figsize=(11, 9))
    disp.plot(ax=ax, cmap='Blues', colorbar=True, xticks_rotation=45)
    ax.set_title(f'Confusion Matrix – {exp_name}', fontsize=12)
    save_path = os.path.join(plots_dir, f'confusion_matrix_{exp_name.replace(" ","_")}.png')
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

    return y_true, y_pred, generator.filenames

print("full_evaluation() ready.")

In [ ]:
def show_misclassified(y_true, y_pred, filenames, test_dir,
                       class_indices, exp_name, plots_dir, n_show=8):
    lbl   = list(class_indices.keys())
    wrong = np.where(y_pred != y_true)[0]
    n     = min(n_show, len(wrong))

    print(f"\n{exp_name} – Misclassification summary")
    print(f"  Total test images    : {len(y_true)}")
    print(f"  Correctly classified : {len(y_true)-len(wrong)}")
    print(f"  Misclassified        : {len(wrong)}")

    if n == 0:
        print("  No misclassified images found.")
        return

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    for i in range(n):
        idx  = wrong[i]
        img  = cv2.imread(os.path.join(test_dir, filenames[idx]))
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img  = cv2.resize(img, (IMG_W, IMG_H))
        axes[i].imshow(img)
        axes[i].set_title(
            f"True : {lbl[y_true[idx]].replace('_',' ')}\n"
            f"Pred : {lbl[y_pred[idx]].replace('_',' ')}",
            fontsize=8, color='darkred')
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')

    save_path = os.path.join(plots_dir, f'misclassified_{exp_name.replace(" ","_")}.png')
    plt.suptitle(f'Misclassified Examples – {exp_name}  (Test Set)', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

print("show_misclassified() ready.")

---
## 7. Experiment 1 – Baseline Architecture (ResNet50)

### Architecture choice and justification

**ResNet50** is chosen as the baseline because:
- It is a well-established standard CNN architecture (He et al., 2016).
- Pre-trained ImageNet weights provide strong feature representations that transfer
  well to fine-grained visual tasks like dog breed classification.
- Transfer learning with **full fine-tuning** (all layers re-trained)
- It provides a clear and systematic starting point

### Assumptions and Intuitions (Experiment 1)

1. **Transfer learning assumption:** ImageNet pre-trained weights capture general
   visual features (edges, textures, shapes) that are highly transferable to
   dog breed classification, reducing the training data required.
2. **Full fine-tuning rationale:** Since the target domain (dog breeds) differs from
   ImageNet's broad categories, re-training all layers allows the model to specialise
   its feature representations for this fine-grained task.
3. **Expected overfitting:** With only ~1,169 training images across 10 classes,
   ResNet50's 24M parameters are likely to overfit. This motivates the customization
   in Experiment 2.
4. **Augmentation strategy:** Random rotation (±20°), shifts (±20%), horizontal flip,
   and zoom (±20%) are used to artificially expand the effective training set size
   and improve generalisation to unseen poses and scales.
   
### Architecture Summary — Experiment 1 (Baseline ResNet50)

| Component | Details |
|:---|:---|
| Backbone | ResNet50 (ImageNet weights, all 175 layers re-trained) |
| Input shape | 224 × 224 × 3 |
| Stage 1 | Conv2D(64, 7×7, s=2) + BN + ReLU + MaxPool(3×3, s=2) |
| Stage 2 (conv2_x) | 3 bottleneck blocks, filters: 64-64-256 |
| Stage 3 (conv3_x) | 4 bottleneck blocks, filters: 128-128-512 |
| Stage 4 (conv4_x) | 6 bottleneck blocks, filters: 256-256-1024 |
| Stage 5 (conv5_x) | 3 bottleneck blocks, filters: 512-512-2048 |
| Activation (blocks) | ReLU after every Conv2D + BatchNorm |
| Global pooling | GlobalAveragePooling2D → 2048-dim vector |
| Dense head | Dense(256, ReLU) → Dropout(0.3) → Dense(10, Softmax) |
| Optimizer | Adam (lr = 1e-4) |
| Loss function | Categorical Cross-Entropy |
| Trainable parameters | 24,114,826 |

### Transfer learning approach

Pre-trained weights initialise the network. The full network is then trained on
the dog breed dataset. This is Transfer Learning Option:
*weights initialised from pre-trained model, entire network updated during training.*

In [ ]:
def build_baseline(input_shape=(IMG_H, IMG_W, 3), num_classes=NUM_CLASSES):
    
    # Load ResNet50 backbone — pre-trained on ImageNet, no top classification layer
    backbone = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )

    backbone.trainable = True
    
    inputs = backbone.input
    x      = backbone.output
    x      = GlobalAveragePooling2D()(x)
    x      = Dense(256, activation='relu')(x)
    x      = Dropout(0.3)(x)
    outputs= Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs, name='Baseline_ResNet50')


baseline_model = build_baseline()
baseline_model.summary()

print(f"\nTotal layers     : {len(baseline_model.layers)}")
print(f"Total parameters : {baseline_model.count_params():,}")
print(f"Trainable params : {sum(tf.size(v).numpy() for v in baseline_model.trainable_variables):,}")

In [ ]:
arch_b = os.path.join(PLOTS_DIR, '02_baseline_architecture.png')
try:
    plot_model(baseline_model, to_file=arch_b,
               show_shapes=True, show_layer_names=True)
    print(f"Architecture diagram saved: {arch_b}")
    display(IPImage(arch_b))
except Exception as e:
    print(f"plot_model skipped ({e})")

In [ ]:
baseline_model.compile(
    optimizer=Adam(learning_rate=LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Baseline model compiled.")
print(f"  Optimiser  : Adam  (lr={LR})")
print(f"  Loss       : categorical_crossentropy")
print(f"  Parameters : {baseline_model.count_params():,}")

### Checkpointing – Baseline

**Storage-efficient strategy:** Only the single best model (lowest `val_loss`) is saved.

- Saves to: `./image_classification/weights/baseline/best_model.keras`
- `EarlyStopping(patience=30)` stops training when `val_loss` stops improving
- `ReduceLROnPlateau` halves the learning rate when `val_loss` stalls for 8 epochs

**Auto-resume:** If the SageMaker session is interrupted, simply re-run the training cell.
The best checkpoint is detected automatically and training continues from there.

In [ ]:

b_ckpt_best = ModelCheckpoint(
    filepath          = os.path.join(B_CKPT_DIR, 'best_model.keras'),
    monitor           = 'val_loss',
    save_best_only    = True,
    save_weights_only = False,
    mode              = 'min',
    verbose           = 1
)
b_early = EarlyStopping(
    monitor              = 'val_loss',
    patience             = PATIENCE,
    restore_best_weights = True,
    verbose              = 1
)
b_reduce_lr = ReduceLROnPlateau(
    monitor  = 'val_loss',
    factor   = 0.5,
    patience = 8,
    min_lr   = 1e-6,
    verbose  = 1
)
b_callbacks = [b_ckpt_best, b_early, b_reduce_lr]
print("Baseline callbacks ready.")
print(f"  Best model only -> {B_CKPT_DIR}/best_model.keras")
print(f"  EarlyStopping   : patience={PATIENCE}, monitor=val_loss")
print(f"  ReduceLROnPlateau: factor=0.5, patience=8, min_lr=1e-6")


### Train or Auto-Resume – Baseline

This single cell handles both cases automatically:
- If **no checkpoints exist** — starts fresh training.
- If **checkpoints exist** — loads the latest one and continues from where training stopped.

In [ ]:
train_gen, val_gen, test_gen = make_generators()

b_best_path = os.path.join(B_CKPT_DIR, 'best_model.keras')

_b_initial_epoch = 0
if os.path.exists(b_best_path):
    print(f"Checkpoint found: best_model.keras")
    baseline_model = tf.keras.models.load_model(b_best_path)
    baseline_model.compile(
        optimizer=Adam(learning_rate=LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    _b_hist_path = os.path.join(B_CKPT_DIR, 'history.json')
    if os.path.exists(_b_hist_path):
        import json as _j
        with open(_b_hist_path) as _f:
            _prev = _j.load(_f)
        _b_initial_epoch = len(_prev.get('accuracy', []))
        print(f"Resuming baseline from epoch {_b_initial_epoch} (EarlyStopping/LR state reset).")
    else:
        print("Resuming baseline from best checkpoint (no history.json — epoch counter reset to 0).")
else:
    print("No checkpoint found – starting fresh baseline training.")

print(f"\nSteps per epoch : {STEPS}  |  Val steps : {VSTEPS}  |  Max epochs : {MAX_EPOCHS}\n")

history_baseline = baseline_model.fit(
    train_gen,
    steps_per_epoch  = STEPS,
    epochs           = MAX_EPOCHS,
    initial_epoch    = _b_initial_epoch,
    validation_data  = val_gen,
    validation_steps = VSTEPS,
    callbacks        = b_callbacks,
    verbose          = 1
)

print("\nExperiment 1 – Training complete.")
print(f"  Epochs trained       : {len(history_baseline.history['accuracy'])}")
print(f"  Final train accuracy : {history_baseline.history['accuracy'][-1]:.4f}")
print(f"  Final val accuracy   : {history_baseline.history['val_accuracy'][-1]:.4f}")


import json as _json
_info_b = {
    'epochs_trained': len(history_baseline.history['accuracy']),
    'best_val_acc':   float(max(history_baseline.history['val_accuracy'])),
    'final_train_acc': float(history_baseline.history['accuracy'][-1]),
    'final_val_acc':  float(history_baseline.history['val_accuracy'][-1]),
}
with open(os.path.join(B_CKPT_DIR, 'training_info.json'), 'w') as _f:
    _json.dump(_info_b, _f)
print(f"Saved training_info.json -> epochs={_info_b['epochs_trained']}, best_val={_info_b['best_val_acc']:.4f}")


# ── Save FULL history so comparison plot works after kernel restart ────────
import json as _json
_hist_b = {
    'accuracy':     [float(x) for x in history_baseline.history['accuracy']],
    'val_accuracy': [float(x) for x in history_baseline.history['val_accuracy']],
    'loss':         [float(x) for x in history_baseline.history['loss']],
    'val_loss':     [float(x) for x in history_baseline.history['val_loss']],
}
with open(os.path.join(B_CKPT_DIR, 'history.json'), 'w') as _f:
    _json.dump(_hist_b, _f)
print(f"Saved history.json ({len(_hist_b['accuracy'])} epochs)")


In [ ]:
b_best_path = os.path.join(B_CKPT_DIR, 'best_model.keras')
if os.path.exists(b_best_path):
    size_mb = os.path.getsize(b_best_path) / 1e6
    print(f"Best baseline model saved: best_model.keras  ({size_mb:.1f} MB)")
else:
    print("No best model saved yet – run training cell first.")


### Experiment 1 – Evaluation
*(Accuracy, curves, classification report, confusion matrix, misclassified)*

In [ ]:
b_best_path = os.path.join(B_CKPT_DIR, 'best_model.keras')
if not os.path.exists(b_best_path):
    raise FileNotFoundError(
        f"No saved model at {b_best_path}. "
        "Run the Baseline training cell first, then re-run this cell.")

eval_b = tf.keras.models.load_model(b_best_path)
print(f"Loaded model from: {b_best_path}")

train_gen, val_gen, test_gen = make_generators()

# Test accuracy
test_gen.reset()
test_loss_b, test_acc_b = eval_b.evaluate(test_gen, verbose=1)

# Val accuracy (used as proxy for best_val_acc / final_val_acc)
val_gen.reset()
val_loss_b, val_acc_b_now = eval_b.evaluate(val_gen, verbose=0)

# Train accuracy on the train set (proxy for final_train_acc)
train_gen_eval = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input
).flow_from_directory(
    TRAIN_DIR, target_size=(IMG_H, IMG_W),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
train_loss_b, train_acc_b_now = eval_b.evaluate(train_gen_eval, verbose=0)

# Expose variables used by downstream cells
final_train_acc_b = train_acc_b_now
final_val_acc_b   = val_acc_b_now
best_val_acc_b    = val_acc_b_now 

print(f"\n  ResNet50 Baseline – Test Results")
print("-" * 40)
print(f"  Train Accuracy (eval) : {final_train_acc_b:.4f}  ({final_train_acc_b*100:.2f}%)")
print(f"  Val   Accuracy (eval) : {final_val_acc_b:.4f}  ({final_val_acc_b*100:.2f}%)")
print(f"  Test  Accuracy        : {test_acc_b:.4f}  ({test_acc_b*100:.2f}%)")
print(f"  Test  Loss            : {test_loss_b:.4f}")


In [ ]:

curve_path_b = os.path.join(PLOTS_DIR, '03_baseline_curves.png')

try:
    plot_history(history_baseline,
                 'Experiment 1 – Baseline ResNet50',
                 curve_path_b)
    print(f"Saved: {curve_path_b}")

except NameError:
    # Fresh kernel — load and display the saved PNG
    if os.path.exists(curve_path_b):
        print(f"Displaying saved plot: {curve_path_b}")
        from IPython.display import Image as IPImage, display
        display(IPImage(curve_path_b))
    else:
        print(f"No saved plot found at {curve_path_b}.")
        print("Run the Baseline training cell first to generate the curves.")


In [ ]:
train_gen, val_gen, test_gen = make_generators()
y_true_b, y_pred_b, fnames_b = full_evaluation(
    eval_b, test_gen, 'Baseline ResNet50', PLOTS_DIR)


In [ ]:
train_gen, val_gen, test_gen = make_generators()
show_misclassified(y_true_b, y_pred_b, fnames_b, TEST_DIR,
                   test_gen.class_indices,
                   'Baseline ResNet50', PLOTS_DIR)


In [ ]:

try:
    gap_b = final_train_acc_b - final_val_acc_b

    print("Baseline – Training Behaviour Analysis")
    print("=" * 55)
    print(f"  Final Training Accuracy   : {final_train_acc_b:.4f}  ({final_train_acc_b*100:.2f}%)")
    print(f"  Final Validation Accuracy : {final_val_acc_b:.4f}  ({final_val_acc_b*100:.2f}%)")
    print(f"  Best Validation Accuracy  : {best_val_acc_b:.4f}  ({best_val_acc_b*100:.2f}%)")
    print(f"  Test Accuracy             : {test_acc_b:.4f}  ({test_acc_b*100:.2f}%)")
    print(f"  Train-Val Gap             : {gap_b:.4f}  ({gap_b*100:.2f}%)")
    print("=" * 55)

    if gap_b > 0.10:
        print("\nDiagnosis: OVERFITTING")
        print("  Training accuracy significantly exceeds validation accuracy.")
        print("  The baseline ResNet50 is overfitting the small dog-breed dataset")
        print("  despite its strong backbone – task-specific CONV blocks are needed.")
        print("\nAction for Experiment 2:")
        print("  Add 2 CONV blocks to learn more dataset-specific features.")
        print("  Add L2 regularisation + Dropout to reduce the train-val gap.")
    elif 0.05 < gap_b <= 0.10:
        print("\nDiagnosis: MILD OVERFITTING")
        print(f"  Train-val gap of {gap_b*100:.2f}% is noticeable but within acceptable range")
        print("  for a small fine-grained dataset (~1,169 training images, 10 classes).")
        print("  The model generalises reasonably well but has started to memorise")
        print("  training-specific patterns beyond what is useful for unseen data.")
        print("\nAction for Experiment 2:")
        print("  Add 2 task-specific CONV blocks on top of the ResNet50 backbone.")
        print("  These layers learn breed-specific low-level features from the dataset,")
        print("  reducing reliance on generic ImageNet representations.")
        print("  L2 regularisation (weight_decay) and increased Dropout are applied")
        print("  to the new CONV blocks to penalise complexity and close the gap.")
    elif final_train_acc_b < 0.60 and final_val_acc_b < 0.60:
        print("\nDiagnosis: UNDERFITTING")
        print("  Both accuracies are low – model lacks capacity for this task.")
        print("\nAction for Experiment 2: Add 2 residual blocks to increase capacity.")
    else:
        print("\nDiagnosis: Good fit.")
        print(f"  Train-val gap of {gap_b*100:.2f}% is small – model generalises well.")
        print("  Action for Experiment 2: Add CONV blocks to further improve feature learning.")

except NameError:
    print("ERROR: Run the Experiment 1 Evaluation cell directly above this cell first.")


---
## 8. Experiment 2 – Customized Architecture (ResNet50 + 2 Extra CONV Blocks)

### Customization justification and assumptions

**Problem identified in Experiment 1:**
The baseline ResNet50 exhibited overfitting — training accuracy clearly exceeded
validation accuracy. While ResNet50 is a powerful backbone, the general ImageNet
features may not fully capture the fine-grained texture differences between dog
breeds (coat patterns, ear shape, snout proportions) without additional
task-specific feature extraction capacity.

**Customization applied:**
Two additional Conv2D blocks are added **after the ResNet50 backbone output**
and **before the classification head**. Each block contains:
`Conv2D → BatchNormalization → ReLU`

This adds **2 CONV layers** directly to the baseline architecture — the required
CONV-layer customization. No new architecture family is introduced; both
experiments use the same ResNet50 backbone.

### Assumptions and Intuitions (Experiment 2)

1. **Task-specific feature learning:** ResNet50 features are learned on 1000-class
   ImageNet. Adding 2 CONV blocks after the backbone allows the network to learn
   fine-grained, dog-breed-specific features (coat textures, ear shapes) that are
   not well-encoded by general ImageNet representations.
2. **Filter progression (512 → 256):** The first block uses 512 filters to expand
   the feature space after ResNet50's 2048-dimensional output, capturing diverse
   breed-specific patterns. The second block (256 filters) compresses these features
   into a more discriminative representation before the classification head.
3. **L2 regularisation:** Applied to both added CONV layers to penalise large weights,
   discouraging memorisation of the small training set. Combined with the lower
   learning rate (5e-5 vs 1e-4), this reduces the train-validation gap.
4. **Lower learning rate:** The custom model uses LR=5e-5 (half the baseline) because
   the larger network (more parameters) is more sensitive to large gradient updates,
   and a lower rate ensures stable convergence of the added layers.
5. **Expected improvement:** The extra CONV layers should narrow the train-val gap
   from Experiment 1 by learning domain-specific features, and improve test accuracy
   by better separating visually similar breeds (e.g., Saluki vs borzoi).

### Architecture comparison

| Component | Baseline (Exp 1) | Customized (Exp 2) |
|:---|:---:|:---:|
| ResNet50 backbone (49 CONV) | ✓ Yes | ✓ Yes |
| ImageNet pre-trained weights | ✓ Yes | ✓ Yes |
| All backbone layers re-trained | ✓ Yes | ✓ Yes |
| Extra Conv Block 1 – 512 filters + BN + ReLU | — | **Added** |
| Extra Conv Block 2 – 256 filters + BN + ReLU | — | **Added** |
| L2 regularisation on added CONV layers | — | **1e-4** |
| GlobalAveragePooling2D | ✓ Yes | ✓ Yes |
| Dense(256, ReLU) | ✓ Yes | ✓ Yes |
| Dropout rate | 0.3 | 0.5 |
| Dense(10, Softmax) | ✓ Yes | ✓ Yes |
| Data augmentation | ✓ Yes | ✓ Yes |
| Optimizer | Adam | Adam |
| Learning rate | 1e-4 | 5e-5 |
| Loss function | Categorical Cross-Entropy | Categorical Cross-Entropy |
| **Extra CONV layers added** | **0** | **+2** |
| **Total CONV layers** | **49** | **51** |
| **Total parameters (approx.)** | **~24.1M** | **~34.3M** |

In [ ]:
L2_REG = 1e-4  

def build_custom(input_shape=(IMG_H, IMG_W, 3), num_classes=NUM_CLASSES):
    

    backbone = ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    backbone.trainable = True  

    inputs = backbone.input
    x      = backbone.output   

    x = Conv2D(512, (3, 3), padding='same',
               kernel_initializer=glorot_uniform(seed=0),
               kernel_regularizer=l2(L2_REG))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)


    x = Conv2D(256, (3, 3), padding='same',
               kernel_initializer=glorot_uniform(seed=0),
               kernel_regularizer=l2(L2_REG))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x       = GlobalAveragePooling2D()(x)
    x       = Dense(256, activation='relu')(x)
    x       = Dropout(0.5)(x)              
    outputs = Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs, name='Custom_ResNet50_Plus2Conv')


custom_model = build_custom()
custom_model.summary()

print(f"\nParameter comparison:")
print(f"  Baseline (ResNet50 only)       : {baseline_model.count_params():>12,}")
print(f"  Custom   (ResNet50 + 2 CONV)   : {custom_model.count_params():>12,}")

In [ ]:
arch_c = os.path.join(PLOTS_DIR, '04_custom_architecture.png')
try:
    plot_model(custom_model, to_file=arch_c,
               show_shapes=True, show_layer_names=True)
    print(f"Architecture diagram saved: {arch_c}")
    display(IPImage(arch_c))
except Exception as e:
    print(f"plot_model skipped ({e})")

In [ ]:
custom_model.compile(
    optimizer=Adam(learning_rate=CUSTOM_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("Customized ResNet50 compiled.")
print(f"  Learning rate : {CUSTOM_LR}  (baseline used {LR})")
print(f"  L2 reg        : {L2_REG} on added CONV layers")
print(f"\nParameter comparison:")
print(f"  Baseline (ResNet50)            : {baseline_model.count_params():>12,}")
print(f"  Custom   (ResNet50 + 2 CONV)   : {custom_model.count_params():>12,}")

### Checkpointing – Customized Model

**Storage-efficient strategy:** Only the single best model (lowest `val_loss`) is saved.


- Saves to: `./image_classification/weights/custom/best_model.keras`
- `EarlyStopping(patience=30)` stops training when `val_loss` stops improving
- `ReduceLROnPlateau` halves the learning rate when `val_loss` stalls for 8 epochs

**Auto-resume:** If the session is interrupted, re-run the callbacks cell then the training cell.
The best checkpoint is detected automatically.

In [ ]:

c_ckpt_best = ModelCheckpoint(
    filepath          = os.path.join(C_CKPT_DIR, 'best_model.keras'),
    monitor           = 'val_loss',
    save_best_only    = True,
    save_weights_only = False,
    mode              = 'min',
    verbose           = 1
)
c_early = EarlyStopping(
    monitor              = 'val_loss',
    patience             = PATIENCE,
    restore_best_weights = True,
    verbose              = 1
)
c_reduce_lr = ReduceLROnPlateau(
    monitor   = 'val_loss',
    factor    = 0.5,
    patience  = 10,
    min_lr    = 1e-6,
    verbose   = 1
)
c_callbacks = [c_ckpt_best, c_early, c_reduce_lr]
print("Custom model callbacks ready.")
print(f"  Best model only  -> {C_CKPT_DIR}/best_model.keras")
print(f"  EarlyStopping    : patience={PATIENCE}, monitor=val_loss")
print(f"  ReduceLROnPlateau: factor=0.5, patience=10, min_lr=1e-6")
print()
print("NOTE: Per-epoch saves removed to prevent storage overflow.")
print("      If training stops, re-run this cell then the training cell.")
print("      The best checkpoint will be loaded automatically.")


### Train or Auto-Resume – Customized Model

Same auto-resume logic: loads latest checkpoint if found, starts fresh if not.

In [ ]:
train_gen, val_gen, test_gen = make_generators()

c_best_path = os.path.join(C_CKPT_DIR, 'best_model.keras')

_c_initial_epoch = 0
if os.path.exists(c_best_path):
    print(f"Checkpoint found: best_model.keras")
    custom_model = tf.keras.models.load_model(c_best_path)
    custom_model.compile(
        optimizer=Adam(learning_rate=CUSTOM_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    _c_hist_path = os.path.join(C_CKPT_DIR, 'history.json')
    if os.path.exists(_c_hist_path):
        import json as _j
        with open(_c_hist_path) as _f:
            _prev = _j.load(_f)
        _c_initial_epoch = len(_prev.get('accuracy', []))
        print(f"Resuming custom from epoch {_c_initial_epoch} (EarlyStopping/LR state reset).")
    else:
        print("Resuming custom from best checkpoint (no history.json — epoch counter reset to 0).")
else:
    print("No checkpoint found – starting fresh custom training.")

print(f"\nSteps per epoch : {STEPS}  |  Val steps : {VSTEPS}  |  Max epochs : {MAX_EPOCHS}")
print(f"  LR            : {CUSTOM_LR}")
print(f"  L2 reg        : {L2_REG} on added CONV layers\n")

history_custom = custom_model.fit(
    train_gen,
    steps_per_epoch  = STEPS,
    epochs           = MAX_EPOCHS,
    initial_epoch    = _c_initial_epoch,
    validation_data  = val_gen,
    validation_steps = VSTEPS,
    callbacks        = c_callbacks,
    verbose          = 1
)
print("\nExperiment 2 – Training complete.")
print(f"  Epochs trained       : {len(history_custom.history['accuracy'])}")
print(f"  Final train accuracy : {history_custom.history['accuracy'][-1]:.4f}")
print(f"  Final val accuracy   : {history_custom.history['val_accuracy'][-1]:.4f}")

import json as _json
_info_c = {
    'epochs_trained': len(history_custom.history['accuracy']),
    'best_val_acc':   float(max(history_custom.history['val_accuracy'])),
    'final_train_acc': float(history_custom.history['accuracy'][-1]),
    'final_val_acc':  float(history_custom.history['val_accuracy'][-1]),
}
with open(os.path.join(C_CKPT_DIR, 'training_info.json'), 'w') as _f:
    _json.dump(_info_c, _f)
print(f"Saved training_info.json -> epochs={_info_c['epochs_trained']}, best_val={_info_c['best_val_acc']:.4f}")

import json as _json
_hist_c = {
    'accuracy':     [float(x) for x in history_custom.history['accuracy']],
    'val_accuracy': [float(x) for x in history_custom.history['val_accuracy']],
    'loss':         [float(x) for x in history_custom.history['loss']],
    'val_loss':     [float(x) for x in history_custom.history['val_loss']],
}
with open(os.path.join(C_CKPT_DIR, 'history.json'), 'w') as _f:
    _json.dump(_hist_c, _f)
print(f"Saved history.json ({len(_hist_c['accuracy'])} epochs)")


In [ ]:
c_best_path = os.path.join(C_CKPT_DIR, 'best_model.keras')
if os.path.exists(c_best_path):
    size_mb = os.path.getsize(c_best_path) / 1e6
    print(f"Best custom model saved: best_model.keras  ({size_mb:.1f} MB)")
else:
    print("No best model saved yet – run training cell first.")


### Experiment 2 – Evaluation

In [ ]:
c_best_path = os.path.join(C_CKPT_DIR, 'best_model.keras')
if not os.path.exists(c_best_path):
    raise FileNotFoundError(
        f"No saved model at {c_best_path}. "
        "Run the Custom training cell first, then re-run this cell.")

eval_c = tf.keras.models.load_model(c_best_path)
print(f"Loaded model from: {c_best_path}")

train_gen, val_gen, test_gen = make_generators()

# Test accuracy
test_gen.reset()
test_loss_c, test_acc_c = eval_c.evaluate(test_gen, verbose=1)

# Val accuracy
val_gen.reset()
val_loss_c, val_acc_c_now = eval_c.evaluate(val_gen, verbose=0)

# Train accuracy
train_gen_eval_c = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input
).flow_from_directory(
    TRAIN_DIR, target_size=(IMG_H, IMG_W),
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
train_loss_c, train_acc_c_now = eval_c.evaluate(train_gen_eval_c, verbose=0)

# Expose variables used by downstream cells
final_train_acc_c = train_acc_c_now
final_val_acc_c   = val_acc_c_now
best_val_acc_c    = val_acc_c_now

print(f"\n  ResNet50 Custom (+2 CONV) – Test Results")
print("-" * 40)
print(f"  Train Accuracy (eval) : {final_train_acc_c:.4f}  ({final_train_acc_c*100:.2f}%)")
print(f"  Val   Accuracy (eval) : {final_val_acc_c:.4f}  ({final_val_acc_c*100:.2f}%)")
print(f"  Test  Accuracy        : {test_acc_c:.4f}  ({test_acc_c*100:.2f}%)")
print(f"  Test  Loss            : {test_loss_c:.4f}")


In [ ]:

curve_path_c = os.path.join(PLOTS_DIR, '05_custom_curves.png')

try:
    # Training ran this session — plot fresh and save
    plot_history(history_custom,
                 'Experiment 2 – Custom ResNet50 (+2 CONV blocks)',
                 curve_path_c)
    print(f"Saved: {curve_path_c}")

except NameError:
    # Fresh kernel — load and display the saved PNG
    if os.path.exists(curve_path_c):
        print(f"Displaying saved plot: {curve_path_c}")
        from IPython.display import Image as IPImage, display
        display(IPImage(curve_path_c))
    else:
        print(f"No saved plot found at {curve_path_c}.")
        print("Run the Custom training cell first to generate the curves.")


In [ ]:
train_gen, val_gen, test_gen = make_generators()
y_true_c, y_pred_c, fnames_c = full_evaluation(
    eval_c, test_gen, 'Custom ResNet50 Plus2Conv', PLOTS_DIR)


In [ ]:
train_gen, val_gen, test_gen = make_generators()
show_misclassified(y_true_c, y_pred_c, fnames_c, TEST_DIR,
                   test_gen.class_indices,
                   'Custom ResNet50 Plus2Conv', PLOTS_DIR)


---
## 9. Results Comparison and Discussion

*(This section provides all tables and figures.)*

### Experimental Settings

| Setting | Value |
|---|---|
| Training samples | 1,169 |
| Validation samples | 250 |
| Test samples | 251 |
| Image size | 224 × 224 (RGB) |
| Batch size | 32 |
| Optimiser | Adam |
| Max epochs | 150 |
| Early stopping patience | 30 (monitors val_loss) |
| Experiment 1 LR | 1e-4 |
| Experiment 2 LR | 5e-5 |
| Augmentation | Rotation ±20°, shift ±20%, H-flip, zoom ±20%, shear ±20° |

In [ ]:
import json as _json

b_best_path = os.path.join(B_CKPT_DIR, 'best_model.keras')
c_best_path = os.path.join(C_CKPT_DIR, 'best_model.keras')

if not os.path.exists(b_best_path) or not os.path.exists(c_best_path):
    print("One or both saved models not found. Run both training cells first.")
else:
    eval_b = tf.keras.models.load_model(b_best_path)
    eval_c = tf.keras.models.load_model(c_best_path)

    train_gen, val_gen, test_gen = make_generators()

    # ── Evaluate all splits ────────────────────────────────────────────────
    test_gen.reset();  test_loss_b,  test_acc_b  = eval_b.evaluate(test_gen,  verbose=0)
    test_gen.reset();  test_loss_c,  test_acc_c  = eval_c.evaluate(test_gen,  verbose=0)
    val_gen.reset();   val_loss_b,   val_acc_b   = eval_b.evaluate(val_gen,   verbose=0)
    val_gen.reset();   val_loss_c,   val_acc_c   = eval_c.evaluate(val_gen,   verbose=0)

    train_gen_eval = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=preprocess_input
    ).flow_from_directory(TRAIN_DIR, target_size=(IMG_H, IMG_W),
                          batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
    _, train_acc_b = eval_b.evaluate(train_gen_eval, verbose=0)

    train_gen_eval2 = tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=preprocess_input
    ).flow_from_directory(TRAIN_DIR, target_size=(IMG_H, IMG_W),
                          batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)
    _, train_acc_c = eval_c.evaluate(train_gen_eval2, verbose=0)

    gap_b = train_acc_b - val_acc_b
    gap_c = train_acc_c - val_acc_c

    b_epochs_path = os.path.join(B_CKPT_DIR, 'training_info.json')
    c_epochs_path = os.path.join(C_CKPT_DIR, 'training_info.json')

    epochs_b_str = "N/A"
    epochs_c_str = "N/A"
    best_val_acc_b_str = f"{val_acc_b*100:.2f}%"
    best_val_acc_c_str = f"{val_acc_c*100:.2f}%"

    if os.path.exists(b_epochs_path):
        with open(b_epochs_path) as _f:
            _info = _json.load(_f)
        epochs_b_str       = str(_info.get('epochs_trained', 'N/A'))
        best_val_acc_b_str = f"{_info.get('best_val_acc', val_acc_b)*100:.2f}%"

    if os.path.exists(c_epochs_path):
        with open(c_epochs_path) as _f:
            _info = _json.load(_f)
        epochs_c_str       = str(_info.get('epochs_trained', 'N/A'))
        best_val_acc_c_str = f"{_info.get('best_val_acc', val_acc_c)*100:.2f}%"

    try:
        epochs_b_str       = str(len(history_baseline.history['accuracy']))
        best_val_acc_b_str = f"{max(history_baseline.history['val_accuracy'])*100:.2f}%"
    except NameError:
        pass
    try:
        epochs_c_str       = str(len(history_custom.history['accuracy']))
        best_val_acc_c_str = f"{max(history_custom.history['val_accuracy'])*100:.2f}%"
    except NameError:
        pass

    params_b = eval_b.count_params()
    params_c = eval_c.count_params()

    W = 72
    print("=" * W)
    print("  IMAGE CLASSIFICATION RESULTS COMPARISON")
    print("=" * W)
    print(f"{'Metric':<35} {'Baseline (Exp 1)':<20} {'Custom (Exp 2)'}")
    print("-" * W)
    print(f"{'Architecture':<35} {'ResNet50 (base)':<20} {'ResNet50 + 2 CONV'}")
    print(f"{'Total CONV layers':<35} {'49 (ResNet50)':<20} {'49 + 2 (custom)'}")
    print(f"{'Trainable parameters':<35} {params_b:<20,} {params_c:,}")
    print("-" * W)
    print(f"{'Epochs trained':<35} {epochs_b_str:<20} {epochs_c_str}")
    print(f"{'Final Training Accuracy':<35} {train_acc_b*100:>18.2f}%  {train_acc_c*100:.2f}%")
    print(f"{'Final Validation Accuracy':<35} {val_acc_b*100:>18.2f}%  {val_acc_c*100:.2f}%")
    print(f"{'Best Validation Accuracy':<35} {best_val_acc_b_str:>19} {best_val_acc_c_str}")
    print(f"{'Test Accuracy':<35} {test_acc_b*100:>18.2f}%  {test_acc_c*100:.2f}%")
    print(f"{'Test Loss':<35} {test_loss_b:>19.4f}  {test_loss_c:.4f}")
    print(f"{'Train-Val Accuracy Gap':<35} {gap_b*100:>18.2f}%  {gap_c*100:.2f}%")
    print("=" * W)

    improvement = (test_acc_c - test_acc_b) * 100
    print(f"\nCustomized model: {improvement:.2f} percentage point improvement in test accuracy.")
    if improvement > 0:
        print(f"Train-Val gap reduced by {(gap_b - gap_c)*100:.2f}pp "
              f"({gap_b*100:.2f}% -> {gap_c*100:.2f}%), indicating better generalisation.")

    test_acc_b_tbl  = test_acc_b
    test_acc_c_tbl  = test_acc_c
    val_acc_b_tbl   = val_acc_b
    val_acc_c_tbl   = val_acc_c
    train_acc_b_tbl = train_acc_b
    train_acc_c_tbl = train_acc_c


In [ ]:

import json as _json

comparison_path = os.path.join(PLOTS_DIR, '07_comparison_curves.png')

def _load_history_json(ckpt_dir):
    p = os.path.join(ckpt_dir, 'history.json')
    if os.path.exists(p):
        with open(p) as _f:
            return _json.load(_f)
    return None

try:
    val_acc_b_h  = history_baseline.history['val_accuracy']
    val_acc_c_h  = history_custom.history['val_accuracy']
    val_loss_b_h = history_baseline.history['val_loss']
    val_loss_c_h = history_custom.history['val_loss']
    train_acc_b_h  = history_baseline.history['accuracy']
    train_acc_c_h  = history_custom.history['accuracy']
    train_loss_b_h = history_baseline.history['loss']
    train_loss_c_h = history_custom.history['loss']
    print("Using in-memory history objects.")
    _source = "live"
except NameError:
    print("History not in memory – loading from saved history.json files...")
    _hb = _load_history_json(B_CKPT_DIR)
    _hc = _load_history_json(C_CKPT_DIR)
    if _hb is None or _hc is None:
        missing = []
        if _hb is None: missing.append(os.path.join(B_CKPT_DIR, 'history.json'))
        if _hc is None: missing.append(os.path.join(C_CKPT_DIR, 'history.json'))
        print("ERROR: Cannot find saved history files:")
        for m in missing: print(f"  {m}")
        print("Re-run the training cells (they now save history.json automatically).")
        raise SystemExit
    val_acc_b_h    = _hb['val_accuracy']
    val_acc_c_h    = _hc['val_accuracy']
    val_loss_b_h   = _hb['val_loss']
    val_loss_c_h   = _hc['val_loss']
    train_acc_b_h  = _hb['accuracy']
    train_acc_c_h  = _hc['accuracy']
    train_loss_b_h = _hb['loss']
    train_loss_c_h = _hc['loss']
    print(f"Loaded: Baseline={len(val_acc_b_h)} epochs, Custom={len(val_acc_c_h)} epochs")
    _source = "json"

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
(ax1, ax2), (ax3, ax4) = axes

ep_b = range(1, len(val_acc_b_h) + 1)
ep_c = range(1, len(val_acc_c_h) + 1)

ax1.plot(ep_b, val_acc_b_h, 'b-', lw=2, label='Baseline (Exp 1)')
ax1.plot(ep_c, val_acc_c_h, 'r-', lw=2, label='Custom (Exp 2)')
ax1.set_title('Validation Accuracy Comparison', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Val Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ep_b, val_loss_b_h, 'b-', lw=2, label='Baseline (Exp 1)')
ax2.plot(ep_c, val_loss_c_h, 'r-', lw=2, label='Custom (Exp 2)')
ax2.set_title('Validation Loss Comparison', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

ax3.plot(ep_b, train_acc_b_h, 'b-', lw=2, label='Baseline (Exp 1)')
ax3.plot(ep_c, train_acc_c_h, 'r-', lw=2, label='Custom (Exp 2)')
ax3.set_title('Training Accuracy Comparison', fontweight='bold')
ax3.set_xlabel('Epoch'); ax3.set_ylabel('Train Accuracy')
ax3.legend(); ax3.grid(True, alpha=0.3)

ax4.plot(ep_b, train_loss_b_h, 'b-', lw=2, label='Baseline (Exp 1)')
ax4.plot(ep_c, train_loss_c_h, 'r-', lw=2, label='Custom (Exp 2)')
ax4.set_title('Training Loss Comparison', fontweight='bold')
ax4.set_xlabel('Epoch'); ax4.set_ylabel('Train Loss')
ax4.legend(); ax4.grid(True, alpha=0.3)

plt.suptitle('Experiment 1 (Baseline ResNet50) vs Experiment 2 (Custom ResNet50+2CONV)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(comparison_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"Saved -> {comparison_path}")

print()
print("=" * 65)
print("  EXPERIMENT RESULTS SUMMARY")
print("=" * 65)
print(f"  {'Metric':<35} {'Exp 1 Baseline':>13} {'Exp 2 Custom':>13}")
print("-" * 65)


try:
    with open(os.path.join(B_CKPT_DIR, 'training_info.json')) as f:
        _ib = json.load(f)
    with open(os.path.join(C_CKPT_DIR, 'training_info.json')) as f:
        _ic = json.load(f)
    print(f"  {'Epochs trained':<35} {_ib.get('epochs_trained','N/A'):>13} {_ic.get('epochs_trained','N/A'):>13}")
    print(f"  {'Best Val Accuracy':<35} {_ib.get('best_val_acc',0)*100:>12.2f}% {_ic.get('best_val_acc',0)*100:>12.2f}%")
except Exception:
    pass

try:
    print(f"  {'Test Accuracy':<35} {test_acc_b_tbl*100:>12.2f}% {test_acc_c_tbl*100:>12.2f}%")
    print(f"  {'Val Accuracy (current eval)':<35} {val_acc_b_tbl*100:>12.2f}% {val_acc_c_tbl*100:>12.2f}%")
    print(f"  {'Train-Val Gap':<35} {(train_acc_b_tbl - val_acc_b_tbl)*100:>12.2f}% {(train_acc_c_tbl - val_acc_c_tbl)*100:>12.2f}%")
except NameError:
    print("  (Run Results Comparison Table cell for per-experiment metrics)")
print("=" * 65)


### Discussion of Results

**Overall performance:**  
Both models achieve strong test accuracy on the 10-class dog breed dataset,
demonstrating that ResNet50 with ImageNet transfer learning is highly effective
even on small datasets (~1,169 training images). The customized model delivers
a **4.78 percentage point improvement** in test accuracy over the baseline,
confirming the value of task-specific architectural customization.

**Experiment 1 – Baseline ResNet50:**  
The baseline achieves **91.63% test accuracy** with a **train-val gap of 6.80%**,
indicating moderate overfitting. The model memorises training-specific patterns
beyond what generalises to unseen images. This is expected given the small dataset
size relative to ResNet50's 24.1M parameters. Despite overfitting, the strong
ImageNet pre-trained features provide a solid baseline for comparison.

**Experiment 2 – Customized ResNet50 (+2 CONV blocks):**  
The customized model achieves **96.41% test accuracy** — a clear improvement over
the baseline. The train-val gap narrows from 6.80% to **5.46%**, and the model
converges faster (35 epochs vs 57 for baseline), confirming that the added
task-specific capacity is both more efficient and better-generalising. The best
validation accuracy also improves from 92.41% to 93.75%.

**Why customization improves performance:**
- The 2 extra CONV blocks (512 → 256 filters) learn fine-grained breed-specific
  features (coat texture, ear shape, snout proportions) not well-captured by the
  ResNet50 backbone trained on diverse 1000-class ImageNet categories.
- L2 regularisation on the added layers penalises overly large weights, preventing
  the new capacity from simply memorising the small training set — directly
  responsible for narrowing the train-val gap by 1.34pp.
- The lower learning rate (5e-5 vs 1e-4) ensures stable fine-tuning of the combined
  backbone + custom head system, with the added layers converging smoothly without
  destabilising the pre-trained ResNet50 weights.

**Misclassified images:**  
The most common misclassifications occur between visually similar breeds:
- *Saluki* vs *borzoi* (both long-snouted sighthounds with silky coats)
- *groenendael* vs *komondor* (both large herding breeds with distinctive coats)
These confusions are expected given the fine-grained nature of the task and the
limited training data per class (~116–146 images per class in training). The
customized model reduces such errors compared to the baseline, reflected in
the higher test accuracy.

**Comparison with literature:**  
ResNet50 fine-tuned on dog breed datasets typically achieves 85–95% accuracy
depending on the number of classes and dataset size. Our baseline result (91.63%)
falls within this range, while the customized model (96.41%) exceeds the upper
bound — attributable to the task-specific CONV blocks learning breed-discriminative
features beyond generic ImageNet representations on this focused 10-class problem.

In [ ]:
all_plots = sorted(glob.glob(os.path.join(PLOTS_DIR, '*.png')))
print(f"Plots saved to {PLOTS_DIR}/:")
for p in all_plots:
    print(f"  {os.path.basename(p)}")